In [1]:
import pandas as pd

movies= pd.read_csv(r"D:\MINI PROJECT\DATASET\movies_extended.csv")

print(movies.shape)
movies.head()


(22515, 17)


,movieId,tmdb_id,title,overview,genres,keywords,director,cast_top5,runtime,release_year,popularity,vote_average,vote_count,poster_path,combined_text,release_date,rating
0,1,862.0,Toy Story,"led by woody, andy's toys live happily in his ...","family, comedy, animation, adventure","rescue, friendship, mission, jealousy, villain...",John Lasseter,"Tom Hanks, Tim Allen, Don Rickles, Jim Varney,...",81.0,1995.0,17.9054,8.000,19336.0,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,"led by woody, andy's toys live happily in his ...",NaN,NaN
1,2,8844.0,Jumanji,when siblings judy and peter discover an encha...,"adventure, fantasy, family","giant insect, board game, disappearance, jungl...",Joe Johnston,"Robin Williams, Kirsten Dunst, Bradley Pierce,...",104.0,1995.0,2.6696,7.243,10985.0,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,when siblings judy and peter discover an encha...,NaN,NaN
2,3,15602.0,Grumpier Old Men,a family wedding reignites the ancient feud be...,"romance, comedy","fishing, sequel, old man, best friend, wedding...",Howard Deutch,"Walter Matthau, Jack Lemmon, Ann-Margret, Soph...",101.0,1995.0,1.9051,6.500,410.0,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,a family wedding reignites the ancient feud be...,NaN,NaN
3,4,31357.0,Waiting to Exhale,"cheated on, mistreated and stepped on, the wom...","comedy, drama, romance","based on novel or book, single mother, divorce...",Forest Whitaker,"Whitney Houston, Angela Bassett, Loretta Devin...",127.0,1995.0,2.3907,6.281,180.0,/qJU6rfil5xLVb5HpJsmmfeSK254.jpg,"cheated on, mistreated and stepped on, the wom...",NaN,NaN
4,5,11862.0,Father of the Bride Part II,just when george banks has recovered from his ...,"comedy, family","daughter, baby, parent child relationship, mid...",Charles Shyer,"Steve Martin, Diane Keaton, Martin Short, Kimb...",106.0,1995.0,2.5283,6.272,780.0,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg,just when george banks has recovered from his ...,NaN,NaN


In [2]:
#Model used-SBERT
#Because convert sentence to vectors and capture semantic meaning

#Loading SBERT model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


d:\MINI PROJECT\venvs\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#Generating movie embeddings based on genre & overview and other text data
texts = movies["combined_text"].fillna("").astype(str).tolist()
movie_embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True
)


Batches: 100%|██████████| 704/704 [10:34<00:00,  1.11it/s]


In [5]:
#Implement cosine similarity

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


In [6]:
def recommend_movies_content_based(
    liked_movie_ids,
    movies_df,
    embeddings,
    top_n=10
):
    # Map movieId → index
    id_to_index = {
        mid: idx for idx, mid in enumerate(movies_df["movieId"])
    }

    liked_indices = [
        id_to_index[mid] for mid in liked_movie_ids
        if mid in id_to_index
    ]

    # Average embedding of liked movies
    user_vector = np.mean(embeddings[liked_indices], axis=0)

    # Compute similarity with all movies
    similarity_scores = cosine_similarity(
        user_vector.reshape(1, -1),
        embeddings
    )[0]

    # Rank movies
    top_indices = similarity_scores.argsort()[::-1]

    # Exclude already liked movies
    recommended_indices = [
        idx for idx in top_indices
        if movies_df.iloc[idx]["movieId"] not in liked_movie_ids
    ][:top_n]

    return movies_df.iloc[recommended_indices][
        ["movieId", "title", "genres"]
    ]


In [7]:
# Example: user liked these movies
liked_movies = [1, 32, 296]  # Toy Story, Twelve Monkeys, Pulp Fiction

recommendations = recommend_movies_content_based(
    liked_movies,
    movies,
    movie_embeddings,
    top_n=10
)

recommendations


,movieId,title,genres
16477,97306,Seven Psychopaths,"comedy, crime"
15889,94112,Twelve,"thriller, drama, action, crime"
13851,82244,13,"drama, thriller"
18953,109742,Cheap Thrills,"comedy, crime, drama, thriller"
3600,4108,Five Corners,"drama, crime, thriller"
14553,86892,The Man from Nowhere,"action, thriller, crime"
9721,47146,Lady Killer,"comedy, crime"
2712,3114,Toy Story 2,"animation, comedy, family"
10776,58146,Witless Protection,"comedy, action, adventure, crime"
1541,1785,King of New York,"thriller, crime"


In [8]:
np.save("D:\MINI PROJECT\DATASET\movie_embeddings.npy", movie_embeddings)
print("Embeddings saved successfully.")

Embeddings saved successfully.


In [9]:
import numpy as np

emb = np.load("D:\MINI PROJECT\DATASET\movie_embeddings.npy")

print(emb.shape)

(22515, 384)


In [11]:
import pandas as pd
import numpy as np

movies = pd.read_csv(r"D:\MINI PROJECT\DATASET\movies_extended.csv")
embeddings = np.load("D:\MINI PROJECT\DATASET\movie_embeddings.npy")

print("Movies:", len(movies))
print("Embeddings:", embeddings.shape)

Movies: 22515
Embeddings: (22515, 384)


In [12]:
liked_movies = [1, 32, 296]

recommendations = recommend_movies_content_based(
    liked_movies,
    movies,
    embeddings,
    top_n=10
)

print(recommendations)

       movieId                 title                            genres
16477    97306     Seven Psychopaths                     comedy, crime
15889    94112                Twelve    thriller, drama, action, crime
13851    82244                    13                   drama, thriller
18953   109742         Cheap Thrills    comedy, crime, drama, thriller
3600      4108          Five Corners            drama, crime, thriller
14553    86892  The Man from Nowhere           action, thriller, crime
9721     47146           Lady Killer                     comedy, crime
2712      3114           Toy Story 2         animation, comedy, family
10776    58146    Witless Protection  comedy, action, adventure, crime
1541      1785      King of New York                   thriller, crime


In [13]:
movies[movies["title"].str.contains("Zootopia 2", na=False)]

,movieId,tmdb_id,title,overview,genres,keywords,director,cast_top5,runtime,release_year,popularity,vote_average,vote_count,poster_path,combined_text,release_date,rating
22021,1084242,NaN,Zootopia 2,After cracking the biggest case in Zootopia's ...,NaN,NaN,NaN,NaN,NaN,2025.0,128.6089,NaN,1968.0,https://image.tmdb.org/t/p/w500/oJ7g2CifqpStmo...,NaN,2025-11-26,7.6


In [14]:
movies["combined_text"].isnull().sum()

np.int64(502)

In [16]:
import pandas as pd
import numpy as np

movies = pd.read_csv("D:/MINI PROJECT/DATASET/movies_final_merged.csv")
emb = np.load("D:/MINI PROJECT/checkpoints/movie_embeddings.npy")

print("Movies:", len(movies))
print("Embeddings:", emb.shape)

Movies: 22515
Embeddings: (22515, 384)
